# REM sleep without atonia: CAP extraction and scoring

Runs entirely in the cloud. Each of CAP's 108 recordings is streamed from
PhysioNet's open mirror (about 40 GB in total) onto this Colab machine,
reduced to a small EMG-envelope file, and the raw file is deleted.

* Steps 1-2 work on the Colab machine's own disk and need no permissions.
* Step 3 copies the results to Google Drive (`MyDrive/rbd-rwa`) so they
  survive the session. It asks for permission to access Drive.
* In a later session, run step 0 first to bring earlier results back, and
  extraction continues where it stopped.

In [ ]:
import os
os.environ['RBD_DURABLE_DIR'] = '/content/durable'
!mkdir -p /content/rbd-rwa/src /content/durable
%cd /content/rbd-rwa

In [ ]:
%%writefile /content/rbd-rwa/src/__init__.py

In [ ]:
%%writefile /content/rbd-rwa/src/config.py
"""Paths that work identically on Colab, locally, or anywhere else.

The guiding rule: RAW recordings are disposable, DERIVED data is precious.
Raw PSG is hundreds of MB per night and lives in scratch (Colab's ephemeral
disk); the extracted envelope + hypnogram is a few MB and lives in a durable
directory (Google Drive), so a disconnected session never loses real work.
"""
import os
from pathlib import Path


def _first_existing(*candidates):
    for c in candidates:
        if c and Path(c).exists():
            return Path(c)
    return None


ON_COLAB = "COLAB_GPU" in os.environ or Path("/content").exists()

# Durable storage: Drive on Colab, the project folder otherwise.
DRIVE = _first_existing(
    os.environ.get("RBD_DRIVE_DIR"),
    "/content/drive/MyDrive",
)
PROJECT_ROOT = Path(__file__).resolve().parent.parent
DURABLE = Path(os.environ.get("RBD_DURABLE_DIR",
                              (DRIVE / "rbd-rwa") if DRIVE else PROJECT_ROOT / "data"))

# Scratch: fast, large, and wiped when the session ends. Raw EDFs only.
SCRATCH = Path(os.environ.get("RBD_SCRATCH_DIR",
                              "/content/scratch" if ON_COLAB else PROJECT_ROOT / "data" / "raw"))

DERIVED   = DURABLE / "derived"      # one compact .npz per recording
MANIFESTS = DURABLE / "manifests"    # progress records, so a restart resumes
FIGURES   = DURABLE / "figures"
RESULTS   = DURABLE / "results"

for d in (DERIVED, MANIFESTS, FIGURES, RESULTS, SCRATCH):
    d.mkdir(parents=True, exist_ok=True)

# Envelope sampling rate. Phasic RWA bursts are >= 0.1 s, so 50 Hz (20 ms bins)
# gives 5 samples across the shortest event a method needs to see, while being
# ~100x smaller than the raw signal.
ENV_FS = 50

# EMG band. 10 Hz high-pass removes movement/drift and ECG baseline; 100 Hz is
# the conventional upper edge, lowered automatically when Nyquist forbids it.
EMG_BAND = (10.0, 100.0)

# A second, common band that every recording can supply. Chin EMG in CAP is
# sampled anywhere from 100 to 512 Hz, and amplitude depends on bandwidth, so
# comparing amplitudes ACROSS recordings needs the same band for all of them.
# 45 Hz is the highest edge a 100 Hz recording allows (Nyquist minus 5).
HARMONIZED_BAND = (10.0, 45.0)


def describe():
    print(f"on Colab : {ON_COLAB}")
    print(f"durable  : {DURABLE}   (derived data, survives disconnects)")
    print(f"scratch  : {SCRATCH}   (raw EDFs, disposable)")

In [ ]:
%%writefile /content/rbd-rwa/src/edf.py
"""Minimal EDF/EDF+ reader that returns each channel at its NATIVE rate.

Why not MNE: `mne.io.read_raw_edf` resamples every channel to the highest rate
in the file. CAP stores chin EMG at 256-512 Hz but leg EMG at 128 Hz, so the
leg channels silently came back upsampled to 512 Hz. Reading the samples
directly keeps each channel's true rate, reads only the channels asked for,
and needs nothing beyond numpy - which matters on hosted notebooks where
installing packages is not always possible.

Values are returned in microvolts when the header's physical dimension is a
voltage. Some files carry no unit at all (raw converter counts); those are
returned unscaled and flagged, because an absolute amplitude threshold means
nothing for them.
"""
from dataclasses import dataclass
from pathlib import Path

import numpy as np

_TO_UV = {"uv": 1.0, "µv": 1.0, "mv": 1e3, "v": 1e6, "nv": 1e-3}


@dataclass
class Channel:
    index: int
    label: str
    dim: str
    fs: float
    n_per_record: int
    gain: float          # physical units per digital unit
    offset: float        # physical value at digital zero
    prefilter: str
    transducer: str

    @property
    def uv_scale(self):
        """Factor from physical units to microvolts, or None if not a voltage."""
        return _TO_UV.get(self.dim.strip().lower().replace("μ", "µ"))


@dataclass
class EdfHeader:
    path: Path
    n_records: int
    record_s: float
    header_bytes: int
    start_date: str
    start_time: str
    channels: list

    @property
    def duration_s(self):
        return self.n_records * self.record_s

    def find(self, label):
        for c in self.channels:
            if c.label == label:
                return c
        raise KeyError(label)


def read_header(path) -> EdfHeader:
    path = Path(path)
    with open(path, "rb") as f:
        h = f.read(256)
        ns = int(h[252:256])
        s = f.read(256 * ns)

    def fields(width, start):
        return [s[start + i * width: start + (i + 1) * width]
                .decode("latin-1").strip() for i in range(ns)], start + width * ns

    labels, o = fields(16, 0)
    trans, o = fields(80, o)
    dims, o = fields(8, o)
    pmin, o = fields(8, o)
    pmax, o = fields(8, o)
    dmin, o = fields(8, o)
    dmax, o = fields(8, o)
    pref, o = fields(80, o)
    nsamp, o = fields(8, o)

    header_bytes = int(h[184:192])
    n_records = int(h[236:244])
    record_s = float(h[244:252])
    per_rec = [int(x) for x in nsamp]
    if n_records < 0:                             # "-1" = unknown; infer it
        data_bytes = path.stat().st_size - header_bytes
        n_records = data_bytes // (2 * sum(per_rec))

    chans = []
    for i in range(ns):
        p0, p1, d0, d1 = (float(pmin[i]), float(pmax[i]),
                          float(dmin[i]), float(dmax[i]))
        gain = (p1 - p0) / (d1 - d0) if d1 != d0 else 1.0
        chans.append(Channel(
            index=i, label=labels[i], dim=dims[i],
            fs=per_rec[i] / record_s if record_s > 0 else float("nan"),
            n_per_record=per_rec[i], gain=gain, offset=p0 - gain * d0,
            prefilter=pref[i], transducer=trans[i]))
    return EdfHeader(path, n_records, record_s, header_bytes,
                     h[168:176].decode("ascii", "ignore").strip(),
                     h[176:184].decode("ascii", "ignore").strip(), chans)


def read_channel(hdr: EdfHeader, channel):
    """
    One channel as float64 at its native rate, in microvolts when the unit is
    a voltage. `channel` is a Channel, an index, or a label (first match).
    Returns (signal, fs, unit_known).
    """
    if isinstance(channel, Channel):
        ch = channel
    elif isinstance(channel, int):
        ch = hdr.channels[channel]
    else:
        ch = hdr.find(channel)
    per_rec = [c.n_per_record for c in hdr.channels]
    rec_len = sum(per_rec)
    start = sum(per_rec[:ch.index])
    raw = np.memmap(hdr.path, dtype="<i2", mode="r", offset=hdr.header_bytes,
                    shape=(hdr.n_records, rec_len))
    x = np.asarray(raw[:, start:start + ch.n_per_record], dtype=np.float64).ravel()
    del raw
    x = x * ch.gain + ch.offset
    k = ch.uv_scale
    if k is not None:
        x *= k
    return x, ch.fs, k is not None

In [ ]:
%%writefile /content/rbd-rwa/src/ecg.py
"""Remove the heartbeat from chin EMG.

The chin electrodes pick up the ECG. Each QRS complex adds a ~50-100 ms
deflection to the chin signal, so in quiet REM sleep - exactly where RWA is
measured - every heartbeat can look like a small muscle burst. In CAP's
healthy controls the chin amplitude at the R-peak is up to 2.5x the
surrounding level, enough to cross the 2x-background threshold of SINBAR.
Automated RWA tools remove it (RBDtector uses template subtraction), and so
do we:

  1. find R-peaks on the ECG channel
  2. average the filtered chin EMG around them, in 5-minute blocks, into a
     heartbeat template (a median, so genuine muscle bursts do not leak in)
  3. subtract the template at every beat
"""
import numpy as np

ECG_BIPOLAR = ("ECG1-ECG2", "EKG1-EKG2")
ECG_PAIRS = (("ECG1", "ECG2"), ("EKG1", "EKG2"))
ECG_PREFIX = ("ECG", "EKG")


def _norm(label):
    return label.upper().replace(" ", "")


def find_ecg(hdr):
    """The ECG source: a channel, a (positive, negative) pair, or None."""
    chans = hdr.channels
    for c in chans:
        if any(p in _norm(c.label) for p in ECG_BIPOLAR):
            return c
    lab = {_norm(c.label): c for c in chans}
    for a, b in ECG_PAIRS:
        if a in lab and b in lab and lab[a].fs == lab[b].fs:
            return (lab[a], lab[b])
    return next((c for c in chans if _norm(c.label).startswith(ECG_PREFIX)), None)


def r_peaks(ecg, fs, window_s=30.0):
    """
    R-peak times in seconds. Band-pass 5-20 Hz, orient so the QRS points up,
    then keep peaks >= 0.3 s apart (<= 200 bpm) that exceed 35% of the local
    99th percentile, computed in `window_s` windows so that slow changes in
    ECG amplitude through the night do not break detection.
    """
    from scipy.signal import butter, sosfiltfilt, find_peaks
    if fs < 50 or len(ecg) < fs * 60:
        return np.empty(0)
    sos = butter(3, [5.0, min(20.0, fs / 2 - 1)], btype="band", fs=fs, output="sos")
    y = sosfiltfilt(sos, np.asarray(ecg, dtype=np.float64))
    if abs(np.percentile(y, 0.1)) > abs(np.percentile(y, 99.9)):
        y = -y
    w = int(window_s * fs)
    n = int(np.ceil(len(y) / w))
    level = np.array([np.percentile(y[i * w:(i + 1) * w], 99) for i in range(n)])
    height = np.repeat(0.35 * level, w)[: len(y)]
    peaks, _ = find_peaks(y, height=height, distance=int(0.3 * fs))
    return peaks / fs


def plausible(peaks_s, duration_s):
    """True when the detected beats look like a heart rate, not noise."""
    if len(peaks_s) < 100:
        return False
    rate = 60.0 * len(peaks_s) / duration_s
    rr = np.diff(peaks_s)
    return 35.0 <= rate <= 130.0 and np.mean((rr > 0.33) & (rr < 2.0)) > 0.9


def subtract_heartbeat(x, fs, peaks_s, pre_s=0.10, post_s=0.15, block_s=300.0):
    """
    Subtract a local heartbeat template from a FILTERED signal. Returns the
    cleaned copy. Segments never overlap because beats are >= 0.3 s apart.
    """
    from scipy.signal.windows import tukey
    x = np.array(x, dtype=np.float64, copy=True)
    a, b = int(round(pre_s * fs)), int(round(post_s * fs))
    idx = np.round(np.asarray(peaks_s) * fs).astype(np.int64)
    idx = idx[(idx - a >= 0) & (idx + b <= len(x))]
    if len(idx) < 20:
        return x
    seg = idx[:, None] + np.arange(-a, b)[None, :]
    beats = x[seg]
    taper = tukey(a + b, alpha=0.3)
    overall = np.median(beats, axis=0)
    block = (idx / fs // block_s).astype(np.int64)
    for k in np.unique(block):
        m = block == k
        tmpl = np.median(beats[m], axis=0) if m.sum() >= 20 else overall
        x[seg[m]] -= tmpl * taper
    return x


def artifact_ratio(env, env_fs, peaks_s):
    """
    Heartbeat-locked envelope profile: the median at each lag around the
    R-peak, peak (-40..+80 ms) over baseline (-500..-300 ms). ~1 means no
    heartbeat artefact; 2.5 means the beat dominates quiet EMG. Medians at
    each lag (not a maximum over noisy single beats) keep this unbiased.
    """
    i = np.round(np.asarray(peaks_s) * env_fs).astype(np.int64)
    lags = np.arange(-int(0.5 * env_fs), int(0.1 * env_fs) + 1)
    i = i[(i + lags[0] >= 0) & (i + lags[-1] < len(env))]
    if len(i) < 50:
        return float("nan")
    prof = np.array([np.median(env[i + k]) for k in lags])
    t = lags / env_fs
    base = np.median(prof[(t >= -0.5) & (t <= -0.3)])
    peak = prof[(t >= -0.04) & (t <= 0.08)].max()
    return float(peak / base) if base > 0 else float("nan")

In [ ]:
%%writefile /content/rbd-rwa/src/fetch.py
"""Resumable downloads. PhysioNet's TLS drops connections mid-transfer, so
every request retries and partial files resume with an HTTP Range header.

Open PhysioNet databases are mirrored byte-for-byte on AWS Open Data
(s3://physionet-open), which is far faster from cloud notebooks than
physionet.org, so the mirror is tried first and PhysioNet is the fallback."""
import time
import urllib.request
from pathlib import Path

CAP_MIRRORS = (
    "https://physionet-open.s3.amazonaws.com/capslpdb/1.0.0/",
    "https://physionet.org/files/capslpdb/1.0.0/",
)
CAP_BASE = CAP_MIRRORS[-1]


def remote_size(url, tries=5):
    for i in range(tries):
        try:
            req = urllib.request.Request(url, method="HEAD")
            return int(urllib.request.urlopen(req, timeout=45)
                       .headers.get("Content-Length", 0))
        except Exception:
            if i == tries - 1:
                return None
            time.sleep(3)


def download(url, dest: Path, tries=10, quiet=False):
    """Fetch `url` to `dest`, resuming if a partial file is already there."""
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    total = remote_size(url)
    if total is None:
        raise IOError(f"cannot reach {url}")
    if dest.exists() and dest.stat().st_size == total:
        return dest
    for i in range(tries):
        have = dest.stat().st_size if dest.exists() else 0
        if have == total:
            return dest
        if have > total:                          # stale file from another source
            dest.unlink()
            have = 0
        try:
            headers = {"Range": f"bytes={have}-"} if have else {}
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=120) as r, open(dest, "ab") as f:
                while chunk := r.read(1 << 20):
                    f.write(chunk)
        except Exception as e:
            if not quiet:
                print(f"    {dest.name}: {type(e).__name__}, resuming "
                      f"({dest.stat().st_size / 1e6 if dest.exists() else 0:.0f} MB)")
            time.sleep(3)
    got = dest.stat().st_size if dest.exists() else 0
    if got != total:
        raise IOError(f"{dest.name}: got {got} of {total} bytes")
    return dest


def download_any(name, dest: Path, mirrors=CAP_MIRRORS):
    """Try each mirror in turn; the first one that completes wins."""
    last = None
    for base in mirrors:
        try:
            return download(base + name, dest, quiet=True)
        except Exception as e:
            last = e
    raise IOError(f"{name}: all mirrors failed ({last})")


def cap_records():
    """All CAP record names, grouped by cohort prefix."""
    txt = None
    for base in CAP_MIRRORS:
        try:
            txt = urllib.request.urlopen(base + "RECORDS", timeout=45).read().decode()
            break
        except Exception:
            continue
    if txt is None:
        raise IOError("cannot fetch the CAP RECORDS list from any mirror")
    recs = [r.replace(".edf", "") for r in txt.split()]
    groups = {}
    for r in recs:
        key = "".join(c for c in r if not c.isdigit())
        groups.setdefault(key, []).append(r)
    return groups


def fetch_cap(record, scratch: Path):
    """Download one CAP recording (.edf + .txt) into scratch."""
    edf = download_any(record + ".edf", Path(scratch) / f"{record}.edf")
    ann = download_any(record + ".txt", Path(scratch) / f"{record}.txt")
    return edf, ann

In [ ]:
%%writefile /content/rbd-rwa/src/cap.py
"""Parse CAP Sleep Database annotations and align them to the EDF clock.

The annotation file is a RemLogic export whose epochs are stamped with WALL
CLOCK times ("23:07:19"), while the signal array starts at whatever time the
EDF header declares. Aligning them is the step most likely to go silently
wrong: an unnoticed offset shifts every sleep stage relative to the signal and
quietly corrupts every downstream measurement, so the offset is computed
explicitly here and reported.
"""
import datetime as dt
import re
from pathlib import Path

import numpy as np

# W / S1 / S2 / S3 / S4 / R / MT  ->  compact codes
STAGE_CODE = {"W": 0, "S1": 1, "S2": 2, "S3": 3, "S4": 3, "R": 4}
STAGE_NAME = {0: "Wake", 1: "N1", 2: "N2", 3: "N3", 4: "REM", -1: "excluded"}
EPOCH_SEC = 30


def edf_start_datetime(edf_path: Path) -> dt.datetime:
    """Recording start from the EDF header (bytes 168-183)."""
    with open(edf_path, "rb") as f:
        head = f.read(184)
    d = head[168:176].decode("ascii", "ignore").strip()   # dd.mm.yy
    t = head[176:184].decode("ascii", "ignore").strip()   # hh.mm.ss
    day, mon, yr = (int(x) for x in re.split(r"[.\-/]", d))
    hh, mm, ss = (int(x) for x in re.split(r"[.\-:]", t))
    # EDF year is two digits: 85-99 -> 19xx, 00-84 -> 20xx (EDF spec clipping rule)
    yr += 1900 if yr >= 85 else 2000
    return dt.datetime(yr, mon, day, hh, mm, ss)


def parse_annotations(txt_path: Path):
    """Return (times, stages) for every scored 30 s epoch, in file order."""
    raw = Path(txt_path).read_bytes().decode("latin-1")
    lines = [l.rstrip("\r") for l in raw.split("\n")]

    # The table begins at the header row; column order varies between files.
    hdr_i = next(i for i, l in enumerate(lines) if l.startswith("Sleep Stage"))
    cols = [c.strip() for c in lines[hdr_i].split("\t")]
    i_stage = cols.index("Sleep Stage")
    i_time = next(i for i, c in enumerate(cols) if c.startswith("Time"))
    i_event = cols.index("Event") if "Event" in cols else None

    times, stages = [], []
    for line in lines[hdr_i + 1:]:
        parts = line.split("\t")
        if len(parts) <= max(i_stage, i_time):
            continue
        # Only sleep-stage rows; CAP files also carry MCAP-A1/A2/A3 arousal events
        if i_event is not None and i_event < len(parts):
            if not parts[i_event].strip().upper().startswith("SLEEP"):
                continue
        stage = parts[i_stage].strip().upper()
        tm = parts[i_time].strip()
        m = re.match(r"(\d{1,2})[:.](\d{2})[:.](\d{2})", tm)
        if not m:
            continue
        times.append(tuple(int(g) for g in m.groups()))
        stages.append(STAGE_CODE.get(stage, -1))
    return times, np.array(stages, dtype=np.int8)


def build_hypnogram(edf_path: Path, txt_path: Path, n_seconds: float):
    """
    Hypnogram on the EDF's own timeline: one code per 30 s epoch from recording
    start. Returns (hypnogram, info).
    """
    start = edf_start_datetime(edf_path)
    times, stages = parse_annotations(txt_path)
    if len(times) == 0:
        raise ValueError(f"{Path(txt_path).name}: no scored epochs found")

    # Walk the clock forward, adding a day whenever it wraps past midnight.
    day = 0
    prev = None
    offsets = []
    for hh, mm, ss in times:
        cur = hh * 3600 + mm * 60 + ss
        if prev is not None and cur < prev - 3600:   # wrapped midnight
            day += 1
        prev = cur
        stamp = dt.datetime(start.year, start.month, start.day, hh, mm, ss) \
            + dt.timedelta(days=day)
        offsets.append((stamp - start).total_seconds())

    offsets = np.array(offsets)
    # The first annotation may precede the EDF start by a day if the recording
    # itself began before midnight and the header date is the following day.
    if offsets[0] < -12 * 3600:
        offsets += 24 * 3600

    n_epochs = int(np.floor(n_seconds / EPOCH_SEC))
    hyp = np.full(n_epochs, -1, dtype=np.int8)
    idx = np.round(offsets / EPOCH_SEC).astype(int)
    keep = (idx >= 0) & (idx < n_epochs)
    hyp[idx[keep]] = stages[keep]

    info = {
        "edf_start": start.isoformat(),
        "first_annotation_offset_s": float(offsets[0]),
        "n_annotated": int(len(stages)),
        "n_placed": int(keep.sum()),
        "n_dropped_outside_signal": int((~keep).sum()),
        "n_epochs": int(n_epochs),
        "counts": {STAGE_NAME[k]: int((hyp == k).sum())
                   for k in sorted(STAGE_NAME) if (hyp == k).sum()},
    }
    return hyp, info

In [ ]:
%%writefile /content/rbd-rwa/src/extract.py
"""Turn a raw overnight PSG into a small derived file.

One CAP recording is ~370 MB; what RWA quantification needs is the EMG
amplitude envelope and the hypnogram, a few MB. Extracting and discarding the
raw signal is what makes the project runnable on hosted notebooks, where disk
is ephemeral and sessions end without warning.

Per recording we keep:
    chin  (5, n_bins) float16 - chin EMG amplitude per 20 ms bin, heartbeat
              removed: RMS and mean-rectified in the native band (10-100 Hz
              or as high as the sampling rate allows), the same two in the
              common 10-45 Hz band shared by every recording, and finally the
              native-band RMS WITHOUT heartbeat removal, for comparison
    legs  (n_legs, n_bins) float16 - leg EMG RMS, native band
    hypnogram (n_epochs,) int8   - 30 s sleep stages on the EDF timeline
    meta  json - channels, native rates, bands, units, alignment

TIMING. Bins are exactly 1/ENV_FS seconds long whatever the source rate, so a
bin index converts to time without drift. (An earlier version used a whole
number of samples per bin; at 512 Hz that made 51.2 bins per second instead
of 50 and shifted the hypnogram by ~11 minutes by the end of a night.)
"""
import json
from pathlib import Path

import numpy as np

from .config import ENV_FS, EMG_BAND, HARMONIZED_BAND, DERIVED
from .cap import build_hypnogram, EPOCH_SEC
from .edf import read_header, read_channel
from .ecg import find_ecg, r_peaks, plausible, subtract_heartbeat, artifact_ratio

# Chin rows, all with the heartbeat removed except the last, which keeps it
# so the effect of heartbeat removal can itself be measured.
CHIN_ROWS = ("rms", "mav", "rms_h", "mav_h", "rms_with_ecg")
F16_MAX = 65000.0     # float16 tops out at 65504; clip (and count) above this

# How chin and leg EMG are labelled across CAP's recording systems. Bipolar
# labels win over split electrodes, which win over a lone referential name.
CHIN_BIPOLAR = ("EMG1-EMG2", "EMG-EMG", "CHIN1-CHIN2")
CHIN_PAIRS = (("EMG1", "EMG2"), ("CHIN1", "CHIN2"))
CHIN_SINGLE = ("CHIN", "MILO", "EMG", "MENT", "SUBM")   # MILO = mylohyoid
LEG_BIPOLAR = ("DX1-DX2", "SX1-SX2")                    # DX/SX = right/left
LEG_CONTAINS = ("TIB",)
LEG_PAIRS = (("DX1", "DX2"), ("SX1", "SX2"))


def _norm(label):
    return label.upper().replace(" ", "")


def _pair(chans, a, b):
    lab = {_norm(c.label): c for c in chans}
    if a in lab and b in lab and lab[a].fs == lab[b].fs:
        return (lab[a], lab[b])
    return None


def find_emg(hdr):
    """
    Locate chin and leg EMG. Each source is either a single channel or a
    (positive, negative) pair to be subtracted. Returns (chin, legs).
    """
    chans = hdr.channels
    chin = next((c for c in chans if _norm(c.label) in CHIN_BIPOLAR
                 or any(p in _norm(c.label) for p in CHIN_BIPOLAR)), None)
    if chin is None:
        chin = next((p for a, b in CHIN_PAIRS if (p := _pair(chans, a, b))), None)
    if chin is None:
        chin = next((c for name in CHIN_SINGLE for c in chans
                     if _norm(c.label) == name), None)

    legs = [c for c in chans if any(p in _norm(c.label) for p in LEG_BIPOLAR)]
    legs += [c for c in chans if any(p in _norm(c.label) for p in LEG_CONTAINS)
             and c not in legs]
    if not legs:
        legs = [p for a, b in LEG_PAIRS if (p := _pair(chans, a, b))]
    return chin, legs


def read_source(hdr, src):
    """Signal for a channel or a (positive, negative) pair of channels."""
    if isinstance(src, tuple):
        a, fs, ka = read_channel(hdr, src[0])
        b, _, kb = read_channel(hdr, src[1])
        return a - b, fs, ka and kb, f"{src[0].label}-{src[1].label}", src[0]
    x, fs, k = read_channel(hdr, src)
    return x, fs, k, src.label, src


def emg_band(fs, band=EMG_BAND):
    """The EMG pass-band actually usable at this sampling rate."""
    lo, hi = band
    hi = min(hi, fs / 2.0 - 5.0)
    if hi <= lo:
        raise ValueError(f"sampling rate {fs} Hz too low for EMG band {band}")
    return lo, hi


def bandpass(sig, fs, band=EMG_BAND, mains=None):
    """Band-pass (and optionally notch) the EMG. Returns (filtered, (lo, hi))."""
    from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt

    lo, hi = emg_band(fs, band)
    sos = butter(4, [lo, hi], btype="band", fs=fs, output="sos")
    x = sosfiltfilt(sos, np.asarray(sig, dtype=np.float64))
    if mains:
        # Mains and its harmonics that fall inside the pass-band
        for f0 in np.arange(mains, hi, mains):
            b, a = iirnotch(f0, Q=30.0, fs=fs)
            x = filtfilt(b, a, x)
    return x, (lo, hi)


def envelope(sig, fs, env_fs=ENV_FS, band=EMG_BAND, mains=None, beats_s=None):
    """
    Band-pass the EMG, subtract the heartbeat if R-peak times are given, then
    reduce it to amplitude per bin of exactly 1/env_fs seconds.
    Returns (rms, mav, (lo, hi)).
    """
    x, (lo, hi) = bandpass(sig, fs, band, mains)
    if beats_s is not None and len(beats_s):
        x = subtract_heartbeat(x, fs, beats_s)

    n_bins = int(np.floor(len(x) * env_fs / fs))
    if n_bins == 0:
        raise ValueError("signal shorter than one envelope bin")
    edges = np.floor(np.arange(n_bins + 1) * fs / env_fs).astype(np.int64)
    counts = np.diff(edges)
    rms = np.sqrt(np.add.reduceat(x * x, edges[:-1]) / counts)
    mav = np.add.reduceat(np.abs(x), edges[:-1]) / counts
    return rms.astype(np.float32), mav.astype(np.float32), (lo, hi)


def _describe(label, ch, fs, unit_known, band, x):
    peak = float(np.max(np.abs(x))) if len(x) else 0.0
    return {"label": label, "fs": fs, "band": list(band),
            "unit": ch.dim, "unit_known": bool(unit_known),
            "transducer": ch.transducer, "prefilter": ch.prefilter,
            # share of samples sitting at the recording's largest magnitude:
            # well above ~1e-4 means the recorder was clipping
            "clipped_frac": float(np.mean(np.abs(x) >= 0.999 * peak)) if peak > 0 else 0.0}


def extract_recording(edf_path: Path, txt_path: Path, record: str,
                      out_dir: Path = None, overwrite=False, mains=50.0):
    """Extract one CAP recording to a compact .npz. Returns the output path."""
    out_dir = Path(out_dir or DERIVED)
    out = out_dir / f"{record}.npz"
    if out.exists() and not overwrite:
        return out

    hdr = read_header(edf_path)
    chin_src, leg_srcs = find_emg(hdr)
    if chin_src is None:
        raise ValueError(f"{record}: no chin EMG channel among "
                         f"{[c.label for c in hdr.channels]}")

    # Heartbeats, for removing the ECG from the chin EMG
    ecg_src = find_ecg(hdr)
    beats, ecg_label = np.empty(0), None
    if ecg_src is not None:
        e, fs_e, _, ecg_label, _ = read_source(hdr, ecg_src)
        beats = r_peaks(e, fs_e)
        if not plausible(beats, len(e) / fs_e):
            beats = np.empty(0)
        del e

    x, fs, unit_known, label, ch = read_source(hdr, chin_src)
    rms, mav, band = envelope(x, fs, mains=mains, beats_s=beats)
    rms_h, mav_h, band_h = envelope(x, fs, band=HARMONIZED_BAND, mains=mains,
                                    beats_s=beats)
    rms_raw, _, _ = envelope(x, fs, mains=mains)
    chin_meta = _describe(label, ch, fs, unit_known, band, x)
    chin_meta["band_harmonized"] = list(band_h)
    chin_meta["ecg"] = {
        "channel": ecg_label, "beats": int(len(beats)),
        "heart_rate_bpm": round(60.0 * len(beats) / (len(x) / fs), 1) if len(beats) else None,
        "artifact_ratio_before": round(artifact_ratio(rms_raw, ENV_FS, beats), 3),
        "artifact_ratio_after": round(artifact_ratio(rms, ENV_FS, beats), 3),
    }
    chin_rows = [rms, mav, rms_h, mav_h, rms_raw]
    del x

    leg_rows, leg_meta = [], []
    for src in leg_srcs:
        x, fs, k, label, ch = read_source(hdr, src)
        r, _, b = envelope(x, fs, mains=mains)
        leg_rows.append(r)
        leg_meta.append(_describe(label, ch, fs, k, b, x))
        del x

    n = min(len(r) for r in chin_rows + leg_rows)   # sources can differ by a bin
    chin = np.stack([r[:n] for r in chin_rows])
    legs = np.stack([r[:n] for r in leg_rows]) if leg_rows \
        else np.zeros((0, n), np.float32)

    n_seconds = n / ENV_FS
    hyp, align = build_hypnogram(edf_path, txt_path, n_seconds)

    meta = {
        "record": record,
        "chin_channel": chin_meta["label"],
        "leg_channels": [m["label"] for m in leg_meta],
        "chin_info": chin_meta,
        "leg_info": leg_meta,
        "chin_rows": list(CHIN_ROWS),
        "env_fs": ENV_FS,
        "mains_notch": mains,
        "duration_s": float(n_seconds),
        "epoch_sec": EPOCH_SEC,
        "alignment": align,
    }
    # Anything above float16's range is an artefact (tens of millivolts);
    # clip it rather than store inf, and record how much there was.
    meta["bins_clipped_f16"] = int((chin > F16_MAX).sum() + (legs > F16_MAX).sum())
    out_dir.mkdir(parents=True, exist_ok=True)
    tmp = out.with_name(out.stem + ".tmp.npz")
    np.savez_compressed(tmp, chin=np.minimum(chin, F16_MAX).astype(np.float16),
                        legs=np.minimum(legs, F16_MAX).astype(np.float16),
                        hypnogram=hyp, meta=json.dumps(meta))
    tmp.replace(out)                             # never leave a half-written file
    return out


def load_derived(path):
    """
    Read a derived file back. Returns a dict with float32 arrays:
    chin_rms, chin_mav, chin_rms_h, chin_mav_h, legs (n_legs, n) and
    hypnogram, plus meta.
    """
    z = np.load(path, allow_pickle=False)
    meta = json.loads(str(z["meta"]))
    chin = z["chin"].astype(np.float32)
    d = {f"chin_{name}": chin[i] for i, name in enumerate(meta["chin_rows"])}
    d["legs"] = z["legs"].astype(np.float32)
    d["hypnogram"] = z["hypnogram"]
    d["meta"] = meta
    return d


def summarise(path):
    d = load_derived(path)
    meta, hyp = d["meta"], d["hypnogram"]
    rem = int((hyp == 4).sum())
    ci = meta["chin_info"]
    legs = "/".join(f"{m['fs']:g}" for m in meta["leg_info"]) or "-"
    unit = "" if ci["unit_known"] else "  NO-UNITS"
    e = ci.get("ecg", {})
    ecg = (f"ECG artefact x{e['artifact_ratio_before']:.2f}->x{e['artifact_ratio_after']:.2f}"
           if e.get("beats") else "no ECG")
    return (f"{meta['record']:<8} {meta['duration_s']/3600:>5.1f} h  "
            f"chin {ci['label']} @ {ci['fs']:g} Hz{unit}  legs {legs} Hz  "
            f"REM {rem} epochs ({rem*0.5:.0f} min)  {ecg}")

In [ ]:
%%writefile /content/rbd-rwa/src/rwa.py
"""The published methods for quantifying REM sleep without atonia (RWA).

Each method reduces one night to one or more percentages (or an index) that are
compared against a diagnostic cut-off. They differ in epoch length, amplitude
thresholds, what counts as "background", and which cut-off applies:

  Montreal  Montplaisir et al., Mov Disord 2010;25:2044 (after Lapierre &
            Montplaisir, Neurology 1992). Tonic: 20 s epochs, >50% of the
            epoch at >= 2x background or > 10 uV; positive at >= 30%.
            Phasic: 2 s mini-epochs containing bursts > 4x background;
            positive at >= 15%.
  SINBAR    Frauscher et al., Sleep 2012;35:835. Phasic bursts 0.1-5 s at
            > 2x background; tonic as Montreal but on 30 s epochs; "any"
            activity in 3 s mini-epochs. Mentalis-only "any" cut-off 18.2%
            (100% specificity). The better-known 32% needs bilateral flexor
            digitorum superficialis EMG, which most recordings lack.
  AASM      AASM scoring manual. Tonic epoch: >= 50% of a 30 s epoch above
            the minimum NREM amplitude. Phasic epoch: >= 5 of its ten 3 s
            mini-epochs contain 0.1-5 s bursts >= 4x background. The manual
            gives no numeric cut-off for "excessive".
  RAI       REM Atonia Index, Ferri et al., J Sleep Res 2008 and Sleep Med
            2010;11:947 ("improved" computation: subtract the minimum of the
            surrounding 60 one-second mini-epochs). RAI < 0.8 is the commonly
            used abnormal range.

Five COMPUTERISED indices (RAI among them) were compared on the same 146
recordings by Cesari et al., Sleep 2018;41(10):zsy133. This module puts the
visual-criteria methods and the RAI on one code path so they can be applied
unchanged to large clinical cohorts.

OPERATIONALISING "BACKGROUND". The visual methods judge background by eye.
Here it is an explicit, named choice (`baseline_amplitude`), because it
changes every downstream percentage. Values still listed in a method's
`verify` tuple are secondary-source readings not yet checked against the
original paper.
"""
from dataclasses import dataclass, field

import numpy as np

REM_CODE = 4
NREM_CODES = (2, 3)


# ══════════════════════════════════════════════════════════════════════════════
#  SHARED MACHINERY
# ══════════════════════════════════════════════════════════════════════════════

def mini_epoch_rms(env, env_fs, seconds, how="rms"):
    """
    Split an envelope into consecutive mini-epochs and summarise each: "rms"
    for an RMS envelope (gives the RMS of the underlying signal), "mean" for a
    mean-rectified envelope (gives its average rectified amplitude).
    """
    w = int(round(env_fs * seconds))
    n = len(env) // w
    if n == 0:
        return np.empty(0), w
    blocks = env[: n * w].reshape(n, w)
    if how == "mean":
        return blocks.mean(axis=1), w
    return np.sqrt((blocks ** 2).mean(axis=1)), w


def stage_mask_at(env_fs, hypnogram, epoch_sec, codes, n_samples):
    """Boolean mask over envelope samples for the given sleep-stage codes."""
    per = int(round(env_fs * epoch_sec))
    mask = np.zeros(n_samples, dtype=bool)
    for i, code in enumerate(hypnogram):
        if code in codes:
            a = i * per
            if a >= n_samples:
                break
            mask[a: min(a + per, n_samples)] = True
    return mask


def baseline_amplitude(env, env_fs, hypnogram, epoch_sec, how, seconds=1.0):
    """
    Background EMG level against which activity is judged, from 1 s windows.

    how = "min_nrem" : lowest window in NREM (the AASM wording; a literal
                       minimum is set by one window, so unstable)
          "p5_nrem"  : 5th percentile in NREM (robust reading of the same)
          "min_rem" / "p5_rem" : the same within REM itself, i.e. the atonic
                       floor of REM - closer to what a visual scorer calls
                       "background" when judging a REM epoch
    """
    codes = NREM_CODES if how.endswith("nrem") else (REM_CODE,)
    m = stage_mask_at(env_fs, hypnogram, epoch_sec, codes, len(env))
    if m.sum() < env_fs * seconds * 10:
        return np.nan
    r, _ = mini_epoch_rms(env[m], env_fs, seconds)
    if len(r) == 0:
        return np.nan
    return float(r.min() if how.startswith("min") else np.percentile(r, 5))


def runs_above(env, env_fs, threshold, min_s=0.1, max_s=None):
    """
    Boolean array marking samples inside runs of consecutive supra-threshold
    samples lasting at least `min_s` (and at most `max_s`, if given). The upper
    bound separates phasic bursts from sustained tonic activity.
    """
    above = env > threshold
    out = np.zeros(len(env), dtype=bool)
    if not above.any():
        return out
    edges = np.diff(np.r_[0, above.astype(np.int8), 0])
    starts = np.flatnonzero(edges == 1)
    stops = np.flatnonzero(edges == -1)
    dur = (stops - starts) / env_fs
    keep = dur >= min_s
    if max_s is not None:
        keep &= dur <= max_s
    for a, b in zip(starts[keep], stops[keep]):
        out[a:b] = True
    return out


def _blocks(mask_idx, width):
    """Consecutive, non-overlapping blocks of `width` indices from an index list."""
    n = len(mask_idx) // width
    return mask_idx[: n * width].reshape(n, width)


# ══════════════════════════════════════════════════════════════════════════════
#  METHOD DEFINITIONS
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class Method:
    name: str
    tonic_epoch_s: float          # tonic scored on epochs of this length
    tonic_mult: float             # tonic threshold, multiples of background
    tonic_abs_uv: float = None    # "... or > X uV": an absolute alternative
    tonic_baseline: str = "p5_rem"
    phasic_mini_s: float = 3.0    # phasic scored on mini-epochs of this length
    phasic_mult: float = 4.0
    phasic_max_s: float = 5.0     # longest burst still counted as phasic
    phasic_baseline: str = "p5_rem"
    any_mini_s: float = None      # SINBAR "any" density on these mini-epochs
    any_mult: float = None
    phasic_epoch_rule: int = None # AASM: >= this many phasic mini-epochs / 30 s
    cutoffs: dict = field(default_factory=dict)   # metric -> % (positive if >=)
    verify: tuple = field(default_factory=tuple)


METHODS = {
    "montreal": Method(
        "Montreal", tonic_epoch_s=20.0, tonic_mult=2.0, tonic_abs_uv=10.0,
        phasic_mini_s=2.0, phasic_mult=4.0, phasic_max_s=15.0,
        cutoffs={"tonic_pct": 30.0, "phasic_pct": 15.0},
        verify=("phasic_max_s",),     # 10 s vs 15 s differs between sources
    ),
    "sinbar": Method(
        "SINBAR", tonic_epoch_s=30.0, tonic_mult=2.0, tonic_abs_uv=10.0,
        phasic_mini_s=3.0, phasic_mult=2.0, phasic_max_s=5.0,
        any_mini_s=3.0, any_mult=2.0,
        cutoffs={"any_pct": 18.2},    # mentalis only, 3 s mini-epochs
    ),
    "aasm": Method(
        "AASM", tonic_epoch_s=30.0, tonic_mult=1.0, tonic_baseline="p5_nrem",
        phasic_mini_s=3.0, phasic_mult=4.0, phasic_max_s=5.0,
        phasic_epoch_rule=5,
        cutoffs={},                   # the manual gives none
    ),
}


def score_night(env, env_fs, hypnogram, epoch_sec, method: Method, unit_known=True):
    """Apply one method to one night's chin RMS envelope."""
    out = {"method": method.name}
    rem = stage_mask_at(env_fs, hypnogram, epoch_sec, (REM_CODE,), len(env))
    out["rem_minutes"] = round(float(rem.sum()) / env_fs / 60, 1)
    if rem.sum() < env_fs * 60:
        out["error"] = "less than one minute of REM"
        return out

    # ── tonic ────────────────────────────────────────────────────────────────
    bt = baseline_amplitude(env, env_fs, hypnogram, epoch_sec, method.tonic_baseline)
    bp = baseline_amplitude(env, env_fs, hypnogram, epoch_sec, method.phasic_baseline)
    if not (np.isfinite(bt) and bt > 0 and np.isfinite(bp) and bp > 0):
        out["error"] = "background could not be estimated"
        return out
    thr_t = method.tonic_mult * bt
    if method.tonic_abs_uv is not None and unit_known:
        thr_t = min(thr_t, method.tonic_abs_uv)      # "2x background OR > 10 uV"
    idx = np.flatnonzero(rem)
    tb = _blocks(idx, int(round(env_fs * method.tonic_epoch_s)))
    tonic = (env[tb] > thr_t).mean(axis=1) > 0.5 if len(tb) else np.empty(0, bool)

    # ── phasic ───────────────────────────────────────────────────────────────
    thr_p = method.phasic_mult * bp
    burst = runs_above(env, env_fs, thr_p, 0.1, method.phasic_max_s)
    pb = _blocks(idx, int(round(env_fs * method.phasic_mini_s)))
    phasic = burst[pb].any(axis=1) if len(pb) else np.empty(0, bool)

    out.update(background_tonic_uV=round(bt, 3), background_phasic_uV=round(bp, 3),
               tonic_threshold_uV=round(thr_t, 3), phasic_threshold_uV=round(thr_p, 3),
               tonic_pct=round(100.0 * tonic.mean(), 2) if len(tonic) else np.nan,
               phasic_pct=round(100.0 * phasic.mean(), 2) if len(phasic) else np.nan)

    # ── AASM: an epoch is phasic when >= 5 of its ten 3 s mini-epochs are ──
    if method.phasic_epoch_rule:
        per = int(round(env_fs * epoch_sec))
        w = int(round(env_fs * method.phasic_mini_s))
        hits = []
        for i, code in enumerate(hypnogram):
            if code != REM_CODE or (i + 1) * per > len(env):
                continue
            seg = burst[i * per:(i + 1) * per]
            hits.append(seg[: (per // w) * w].reshape(-1, w).any(axis=1).sum()
                        >= method.phasic_epoch_rule)
        out["phasic_epoch_pct"] = round(100.0 * np.mean(hits), 2) if hits else np.nan

    # ── SINBAR "any": tonic, phasic or both, above the same amplitude ──────
    if method.any_mini_s:
        active = runs_above(env, env_fs, method.any_mult * bp, 0.1, None)
        ab = _blocks(idx, int(round(env_fs * method.any_mini_s)))
        out["any_pct"] = round(100.0 * active[ab].any(axis=1).mean(), 2) \
            if len(ab) else np.nan

    hits = [out.get(k, np.nan) >= v for k, v in method.cutoffs.items()
            if np.isfinite(out.get(k, np.nan))]
    out["positive"] = bool(any(hits)) if hits else None
    return out


def rem_atonia_index(env_mav, env_fs, hypnogram, epoch_sec, bin_width=1.0,
                     n_bins=20, noise="moving_min", window_s=60):
    """
    REM Atonia Index. Average rectified amplitude per 1 s mini-epoch of REM,
    noise-corrected, then binned by amplitude:

        RAI = A0 / (1 - A1)

    with A0 the fraction of mini-epochs <= 1 uV and A1 the fraction in
    (1, 2] uV. Runs from 0 (no atonia) to 1 (complete atonia). `env_mav` must
    be a MEAN-RECTIFIED envelope in microvolts, because the bins are absolute.

    noise = "moving_min": subtract the minimum of the `window_s` mini-epochs
            surrounding each one (Ferri et al. 2010, "improved" RAI);
            "global_min": subtract the night's single minimum (2008 version).
    """
    rem = stage_mask_at(env_fs, hypnogram, epoch_sec, (REM_CODE,), len(env_mav))
    if rem.sum() < env_fs * 60:
        return {"method": "RAI", "error": "less than one minute of REM"}
    amp, _ = mini_epoch_rms(env_mav[rem], env_fs, 1.0, how="mean")
    if noise == "moving_min":
        from scipy.ndimage import minimum_filter1d
        amp = amp - minimum_filter1d(amp, size=int(window_s), mode="nearest")
    else:
        amp = amp - amp.min()
    edges = np.arange(0, (n_bins + 1) * bin_width, bin_width)
    hist, _ = np.histogram(amp, bins=np.r_[edges, np.inf])
    frac = hist / hist.sum()
    a0, a1 = float(frac[0]), float(frac[1])
    rai = a0 / (1.0 - a1) if (1.0 - a1) > 1e-9 else float("nan")
    return {
        "method": "RAI",
        "rai": round(rai, 4),
        "frac_silent": round(a0, 4),
        "rem_minutes": round(float(rem.sum()) / env_fs / 60, 1),
        "positive": bool(rai < 0.8) if np.isfinite(rai) else None,
    }


def score_all(d, band="native"):
    """
    Run every method on one derived recording (the dict from load_derived).
    band = "native" uses each recording's own EMG band; "harmonized" uses the
    common 10-45 Hz band, the fair choice when comparing ACROSS recordings made
    on different equipment.
    """
    meta = d["meta"]
    env_fs, epoch_sec, hyp = meta["env_fs"], meta["epoch_sec"], d["hypnogram"]
    if band == "native_with_ecg":            # heartbeat left in: a sensitivity check
        chin_rms, chin_mav = d["chin_rms_with_ecg"], None
    else:
        sfx = "" if band == "native" else "_h"
        chin_rms, chin_mav = d[f"chin_rms{sfx}"], d[f"chin_mav{sfx}"]
    unit_known = meta["chin_info"]["unit_known"]
    rows = [score_night(chin_rms, env_fs, hyp, epoch_sec, m, unit_known)
            for m in METHODS.values()]
    if chin_mav is None:
        rows.append({"method": "RAI", "error": "not computed for this band"})
    elif unit_known:
        rows.append(rem_atonia_index(chin_mav, env_fs, hyp, epoch_sec))
    else:
        rows.append({"method": "RAI", "error": "chin EMG has no physical units"})
    for r in rows:
        r["band"] = band
    return rows

In [ ]:
%%writefile /content/rbd-rwa/run_extract.py
"""Stream CAP recordings: download -> extract -> discard raw.

Designed for hosted notebooks, where the session can vanish at any moment.
Progress is written to a manifest after every recording, so re-running
continues from where it stopped and never re-downloads finished work. The next
few recordings download in the background while the current one is being
extracted, and raw EDFs are deleted as soon as their derived file exists, so
disk use stays flat no matter how many recordings are processed.

    python run_extract.py                  # every CAP record (default)
    python run_extract.py rbd1 rbd2 n1     # specific records
    python run_extract.py --cohorts rbd n  # whole cohorts by prefix
    python run_extract.py --keep-raw       # keep EDFs (local debugging only)
"""
import argparse
import json
import sys
import time
import traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from src import config
from src.fetch import fetch_cap, cap_records
from src.extract import extract_recording, summarise

MANIFEST = config.MANIFESTS / "cap_extract.json"
PREFETCH = 3          # recordings downloading ahead of the one being extracted


def load_manifest():
    if MANIFEST.exists():
        try:
            return json.loads(MANIFEST.read_text())
        except json.JSONDecodeError:
            pass
    return {"done": {}, "failed": {}}


def save_manifest(m):
    tmp = MANIFEST.with_suffix(".tmp")
    tmp.write_text(json.dumps(m, indent=2))
    tmp.replace(MANIFEST)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("records", nargs="*")
    ap.add_argument("--cohorts", nargs="*", help="cohort prefixes, e.g. rbd n")
    ap.add_argument("--keep-raw", action="store_true")
    ap.add_argument("--overwrite", action="store_true")
    args = ap.parse_args()

    config.describe()
    groups = cap_records()
    if args.records:
        records = args.records
    elif args.cohorts:
        records = [r for c in args.cohorts for r in groups.get(c, [])]
    else:
        records = [r for g in groups.values() for r in g]
    print(f"\n{len(records)} records queued "
          f"(cohorts available: { {k: len(v) for k, v in groups.items()} })\n")

    m = load_manifest()
    todo = [r for r in records
            if r not in m["done"] or args.overwrite
            or not (config.DERIVED / f"{r}.npz").exists()]
    print(f"{len(m['done'])} already done, {len(todo)} to process\n")

    pool = ThreadPoolExecutor(max_workers=PREFETCH)
    pending = {}

    def prefetch(i):
        if i < len(todo) and i not in pending:
            pending[i] = pool.submit(fetch_cap, todo[i], config.SCRATCH)

    for i in range(PREFETCH):
        prefetch(i)

    t_start = time.time()
    for i, rec in enumerate(todo):
        prefetch(i + PREFETCH)
        t0 = time.time()
        print(f"[{i + 1}/{len(todo)}] {rec}", flush=True)
        edf = txt = None
        try:
            edf, txt = pending.pop(i).result()
            t_dl = time.time() - t0
            out = extract_recording(edf, txt, rec, overwrite=args.overwrite)
            print(f"    {summarise(out)}  (waited {t_dl:.0f}s for download, "
                  f"{time.time() - t0 - t_dl:.0f}s to extract)", flush=True)
            m["done"][rec] = {"file": out.name, "seconds": round(time.time() - t0, 1)}
            m["failed"].pop(rec, None)
        except Exception as e:
            print(f"    FAILED: {type(e).__name__}: {e}", flush=True)
            traceback.print_exc(limit=2)
            m["failed"][rec] = f"{type(e).__name__}: {e}"
        finally:
            if not args.keep_raw:
                # Raw is disposable once the derived file exists (or failed).
                for p in (edf, txt):
                    if p is not None:
                        Path(p).unlink(missing_ok=True)
        save_manifest(m)
    pool.shutdown(wait=True)

    derived = sorted(p for p in config.DERIVED.glob("*.npz") if ".tmp" not in p.name)
    size = sum(p.stat().st_size for p in derived)
    print(f"\n{'=' * 70}")
    print(f"  {len(m['done'])} recordings extracted, {len(m['failed'])} failed "
          f"in {(time.time() - t_start) / 60:.1f} min")
    print(f"  derived data: {len(derived)} files, {size / 1e6:.0f} MB total")
    print(f"  location: {config.DERIVED}")
    if m["failed"]:
        print("\n  failures:")
        for k, v in m["failed"].items():
            print(f"    {k}: {v}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_score.py
"""Score every derived recording with every RWA method.

Writes one tidy row per (recording, method, band) to
results/cap_rwa_scores.csv, then prints, for every metric, the median in each
cohort and how well it separates RBD from two comparison groups (AUC, 0.5 =
chance):

  healthy        CAP's healthy sleepers. CAUTION: most were recorded on
                 different equipment from the RBD patients (chin EMG sampled
                 at 100-200 Hz, some without physical units), so amplitude
                 differences are partly hardware differences.
  same-system    other CAP patients (epilepsy, PLM, insomnia, narcolepsy...)
                 whose chin EMG was recorded like the RBD patients' (512 Hz,
                 microvolts). The fairer comparison.

    python run_score.py
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config
from src.extract import load_derived
from src.rwa import score_all

COHORT_NAMES = {"rbd": "RBD", "n": "healthy", "nfle": "NFLE", "plm": "PLM",
                "ins": "insomnia", "narco": "narcolepsy", "sdb": "SDB",
                "brux": "bruxism"}
METRICS = ["tonic_pct", "phasic_pct", "phasic_epoch_pct", "any_pct", "atonia_loss"]


def cohort_of(record):
    return COHORT_NAMES.get("".join(c for c in record if not c.isdigit()), "other")


def auc(pos, neg):
    """Probability a random positive outscores a random negative (ties = 1/2)."""
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    pos, neg = pos[np.isfinite(pos)], neg[np.isfinite(neg)]
    if len(pos) == 0 or len(neg) == 0:
        return float("nan"), len(pos), len(neg)
    diff = pos[:, None] - neg[None, :]
    return float(((diff > 0) + 0.5 * (diff == 0)).mean()), len(pos), len(neg)


def main():
    files = sorted(p for p in config.DERIVED.glob("*.npz") if ".tmp" not in p.name)
    print(f"{len(files)} derived recordings in {config.DERIVED}\n")
    rows = []
    for p in files:
        d = load_derived(p)
        meta = d["meta"]
        ci = meta["chin_info"]
        for band in ("native", "harmonized", "native_with_ecg"):
            for r in score_all(d, band=band):
                r.update(record=meta["record"], cohort=cohort_of(meta["record"]),
                         chin_fs=ci["fs"], chin_label=ci["label"],
                         unit_known=ci["unit_known"], n_legs=len(meta["leg_info"]))
                rows.append(r)
    df = pd.DataFrame(rows)
    if "rai" in df:
        df["atonia_loss"] = 1 - df["rai"]              # higher = more RWA, like the rest
    out = config.RESULTS / "cap_rwa_scores.csv"
    df.to_csv(out, index=False)
    print(f"saved {len(df)} rows -> {out}\n")

    same_system = (df.chin_fs == 512) & df.unit_known.astype(bool)
    order = [c for c in COHORT_NAMES.values() if c in set(df["cohort"])]
    for band, g in df.groupby("band"):
        print(f"=== {band} band " + "=" * 60)
        print(f"{'method':<9} {'metric':<17} " + " ".join(f"{c[:8]:>8}" for c in order)
              + f" | {'AUC v healthy':>14} {'AUC v same-sys':>15}")
        for m, gm in g.groupby("method", sort=False):
            ss = same_system.loc[gm.index]
            for metric in METRICS:
                if metric not in gm or gm[metric].notna().sum() == 0:
                    continue
                med = gm.groupby("cohort")[metric].median()
                a1, n1, h1 = auc(gm.loc[gm.cohort == "RBD", metric],
                                 gm.loc[gm.cohort == "healthy", metric])
                a2, n2, h2 = auc(gm.loc[ss & (gm.cohort == "RBD"), metric],
                                 gm.loc[ss & ~gm.cohort.isin(["RBD", "healthy"]), metric])
                print(f"{m:<9} {metric:<17} "
                      + " ".join(f"{med.get(c, np.nan):>8.2f}" for c in order)
                      + f" | {a1:>6.3f} ({n1}v{h1}) {a2:>7.3f} ({n2}v{h2})")
        pos = g.dropna(subset=["positive"]).pivot_table(
            index="method", columns="cohort", values="positive", aggfunc="mean")
        print("\nshare classified POSITIVE by each method's published cut-off:")
        print((100 * pos.reindex(columns=order)).round(0).to_string())
        print()


if __name__ == "__main__":
    main()

## 0. (Later sessions only) Restore earlier results from Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/rbd-rwa && rsync -a /content/drive/MyDrive/rbd-rwa/ /content/durable/ && du -sh /content/durable

## 1. Extract every CAP recording

In [ ]:
!cd /content/rbd-rwa && python run_extract.py

## 2. Score every recording with every RWA method

In [ ]:
!cd /content/rbd-rwa && python run_score.py

## 3. Save results to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/rbd-rwa && rsync -a /content/durable/ /content/drive/MyDrive/rbd-rwa/ && du -sh /content/drive/MyDrive/rbd-rwa